# Import Package

In [ ]:
import os
import re
import shutil
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr


# Import Google Drive Data

In [ ]:
from google.colab import files
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
zip_path = "/content/drive/MyDrive/aiv/gee_data.zip"
extract_to = "/content/gee_data"

os.makedirs(extract_to, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
  zip_ref.extractall(extract_to)

os.listdir(extract_to)

for root, dirs, files in os.walk("/content/gee_data"):
    for file in files[::]:
        print(os.path.join(root, file))

/content/gee_data/gee_data/land_cover_2016_2022.csv
/content/gee_data/gee_data/EU_2016_2022_land_cover_and_climate_data_containing_missing_values.csv
/content/gee_data/gee_data/era5_2016_2022/2021_median_combined_result.csv
/content/gee_data/gee_data/era5_2016_2022/2016_median_combined_result.csv
/content/gee_data/gee_data/era5_2016_2022/2020_median_combined_result.csv
/content/gee_data/gee_data/era5_2016_2022/2017_median_combined_result.csv
/content/gee_data/gee_data/era5_2016_2022/2018_median_combined_result.csv
/content/gee_data/gee_data/era5_2016_2022/2022_median_combined_result.csv
/content/gee_data/gee_data/era5_2016_2022/2019_median_combined_result.csv


In [ ]:
zip_path = "/content/drive/MyDrive/eu-aiv-analysis/ebird_filtered_checklist.zip"
extract_to = "/content/checklist"

os.makedirs(extract_to, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
  zip_ref.extractall(extract_to)

os.listdir(extract_to)

for root, dirs, files in os.walk("/content/checklist"):
    for file in files[::]:
        print(os.path.join(root, file))

/content/checklist/ebird_filtered_checklist/^Passer montanus_filtered_2019to2022.csv
/content/checklist/ebird_filtered_checklist/^Corvus corax_filtered_2019to2022.csv
/content/checklist/ebird_filtered_checklist/^Falco tinnunculus_filtered2019to2022.csv
/content/checklist/ebird_filtered_checklist/^Calidris alpina_filtered_2019to2022.csv
/content/checklist/ebird_filtered_checklist/^Falco peregrinus_filtered2019to2022.csv
/content/checklist/ebird_filtered_checklist/^Aythya fuligula_filtered_2019to2022.csv
/content/checklist/ebird_filtered_checklist/^Spatula clypeata_filtered_2019to2022.csv
/content/checklist/ebird_filtered_checklist/^Larus marinus_filtered_2019to2022.csv
/content/checklist/ebird_filtered_checklist/^Pluvialis squatarola_filtered_2019to2022.csv
/content/checklist/ebird_filtered_checklist/^Numenius arquata_filtered_2019to2022.csv
/content/checklist/ebird_filtered_checklist/^Aythya ferina_filtered_2019to2022.csv
/content/checklist/ebird_filtered_checklist/^Corvus cornix_filte

In [ ]:
!cp -r "/content/drive/MyDrive/aiv/EU_100km_fishnet_simple_by_distance" "/content/EU_100km_fishnet_simple_by_distance"

In [ ]:
lc_path = "/content/gee_data/gee_data/land_cover_2016_2022.csv"
era5_2021_path = "/content/gee_data/gee_data/era5_2016_2022/2021_median_combined_result.csv"
era5_2022_path = "/content/gee_data/gee_data/era5_2016_2022/2022_median_combined_result.csv"

fishnet_path = "/content/EU_100km_fishnet_simple_by_distance"

for p in [lc_path, era5_2021_path, era5_2022_path, fishnet_path]:
  print(p, os.path.exists(p))

/content/gee_data/gee_data/land_cover_2016_2022.csv True
/content/gee_data/gee_data/era5_2016_2022/2021_median_combined_result.csv True
/content/gee_data/gee_data/era5_2016_2022/2022_median_combined_result.csv True
/content/EU_100km_fishnet_simple_by_distance True


# Environment Data Processing

In [ ]:
lc = pd.read_csv(lc_path)
era5_2021 = pd.read_csv(era5_2021_path)
era5_2022 = pd.read_csv(era5_2022_path)

era5_2021 = era5_2021.rename(columns={"Month": "month_number"})
era5_2022 = era5_2022.rename(columns={"Month": "month_number"})
era5_2021["month_number"] = pd.to_datetime(era5_2021["month_number"]).dt.month
era5_2022["month_number"] = pd.to_datetime(era5_2022["month_number"]).dt.month
era5_2022["year_number"] = 2022
era5_2021["year_number"] = 2021

In [ ]:
era5_all = pd.concat([era5_2021, era5_2022], ignore_index=True)
print(len(era5_2021))
print(len(era5_2022))
print(len(era5_all))

27084
27084
54168


In [ ]:
env = pd.merge(lc, era5_all, on=["Id","month_number", "year_number"], how="inner")
print(len(lc))
print(len(era5_all))
print(len(env))
print(len(env[env["year_number"] == 2021]))
print(len(env[env["year_number"] == 2022]))
#env.isna().sum()

189420
54168
54120
27060
27060


# Import eBird Checklist

In [ ]:
ebird_checklist_paths = []

for root, dirs, files in os.walk("/content/checklist"):
    for file in files[::]:
        ebird_checklist_paths.append(os.path.join(root, file))

In [ ]:
# r8: "Ichthyaetus melanocephalus", "Larus canus", "Larus fuscus", "Larus marinus", "Larus michahellis"
# r9: "Limosa lapponica", "Limosa limosa", "Mareca penelope", "Mareca strepera", "Motacilla alba"
# r10: "Numenius arquata", "Passer domesticus", "Passer montanus", "Phalacrocorax carbo", "Phasianus colchicus"
# r11: "Phoenicopterus roseus", "Platalea leucorodia", "Plegadis falcinellus", "Pluvialis squatarola", "Podiceps cristatus"
# r12: "Recurvirostra avosetta", "Somateria mollissima", "Spatula clypeata", "Streptopelia decaocto", "Sturnus vulgaris"
# r13: "Tachybaptus ruficollis", "Tadorna tadorna", "Tringa ochropus", "Tringa totanus", "Turdus merula"
# r14: "Vanellus vanellus", "Anas crecca"

In [ ]:
# r1: "Accipiter nisus", "Aix galericulata", "Anas acuta", "Anas platyrhynchos", "Anser anser"
# r2: "Ardea alba", "Ardea cinerea", "Arenaria interpres", "Athene noctua", "Aythya ferina"
# r3: "Aythya fuligula", "Branta bernicla", "Branta canadensis", "Bucephala clangula", "Buteo buteo"
# r4: "Calidris alba", "Calidris alpina", "Charadrius hiaticula", "Chroicocephalus ridibundus", "Ciconia ciconia"
# r5: "Columba livia", "Columba palumbus", "Corvus corax", "Corvus cornix", "Cygnus cygnus"
# r6: "Cygnus olor", "Egretta garzetta", "Falco peregrinus", "Falco tinnunculus", "Fringilla montifringilla"
# r7: "Fulica atra", "Gallinago gallinago", "Gallinula chloropus", "Gyps fulvus", "Hirundo rustica"

In [ ]:
# "Vanellus vanellus", "Anas crecca", "Branta canadensis", "Numenius arquata", "Larus canus", "Chroicocephalus ridibundus"

In [ ]:
BIRD_LIST = [
    "Ichthyaetus melanocephalus", "Larus canus", "Larus fuscus", "Larus marinus", "Larus michahellis",
    "Limosa lapponica", "Limosa limosa", "Mareca penelope", "Mareca strepera", "Motacilla alba",
    "Numenius arquata", "Passer domesticus", "Passer montanus", "Phalacrocorax carbo", "Phasianus colchicus",
    "Phoenicopterus roseus", "Platalea leucorodia", "Plegadis falcinellus", "Pluvialis squatarola", "Podiceps cristatus",
    "Recurvirostra avosetta", "Somateria mollissima", "Spatula clypeata", "Streptopelia decaocto", "Sturnus vulgaris",
    "Tachybaptus ruficollis", "Tadorna tadorna", "Tringa ochropus", "Tringa totanus", "Turdus merula",
    "Vanellus vanellus", "Anas crecca"
]

RUN_ALL_BIRDS = False

OUTPUT_ROOT = "/content/drive/MyDrive/gpboost_batch_outputs"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

ZIP_EACH_BIRD = True
DOWNLOAD_EACH_BIRD_ZIP = False
SKIP_FINISHED_BIRDS = False

RESPONSE_TRANSFORM = "count"      # "count" or "log1p"
GPBOOST_LIKELIHOOD = "negative_binomial"  # "negative_binomial" or "gaussian"

TRAIN_SEED = 123
BOOSTING_SEED = 42
NUM_BOOST_ROUND = 700

GPBOOST_PARAMS = {
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": -1,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.7,
    "lambda_l1": 0.0,
    "lambda_l2": 1.0,
    "bagging_freq": 0,
    "verbose": 0,
    "seed": BOOSTING_SEED,
}


def safe_filename(name):
    name = str(name).strip().replace("^", "")
    name = re.sub(r"[^A-Za-z0-9._-]+", "_", name)
    return name.strip("_") or "unknown_bird"


def birdname_from_path(path):
    stem = os.path.basename(path)
    stem = stem.replace("^", "")
    stem = re.sub(r"_filtered.*$", "", stem)
    stem = re.sub(r"\.csv$", "", stem)
    return stem


def find_bird_checklist_paths(ebird_checklist_paths, bird_list=None, run_all=False):
    csv_paths = [p for p in ebird_checklist_paths if p.lower().endswith(".csv")]

    if run_all:
        return {birdname_from_path(p): p for p in sorted(csv_paths)}

    selected = {}
    for bird in bird_list:
        bird_clean = str(bird).replace("^", "").lower()
        matches = [
            p for p in csv_paths
            if bird_clean in os.path.basename(p).replace("^", "").lower()
        ]
        if len(matches) == 0:
            print(f"WARNING: ??? bird checklist: {bird}")
            continue
        selected[str(bird).replace("^", "")] = sorted(matches)[0]

    return selected


bird_path_map = find_bird_checklist_paths(
    ebird_checklist_paths=ebird_checklist_paths,
    bird_list=BIRD_LIST,
    run_all=RUN_ALL_BIRDS,
)

print("Number of birds selected:", len(bird_path_map))
for bird, path in bird_path_map.items():
    print(bird, "->", path)


Number of birds selected: 32
Ichthyaetus melanocephalus -> /content/checklist/ebird_filtered_checklist/^Ichthyaetus melanocephalus_filtered_2019to2022.csv
Larus canus -> /content/checklist/ebird_filtered_checklist/^Larus canus_filtered_2019to2022.csv
Larus fuscus -> /content/checklist/ebird_filtered_checklist/^Larus fuscus_filtered_2019to2022.csv
Larus marinus -> /content/checklist/ebird_filtered_checklist/^Larus marinus_filtered_2019to2022.csv
Larus michahellis -> /content/checklist/ebird_filtered_checklist/^Larus michahellis_filtered_2019to2022.csv
Limosa lapponica -> /content/checklist/ebird_filtered_checklist/^Limosa lapponica_filtered_2019to2022.csv
Limosa limosa -> /content/checklist/ebird_filtered_checklist/^Limosa limosa_filtered_2019to2022.csv
Mareca penelope -> /content/checklist/ebird_filtered_checklist/^Mareca penelope_filtered_2019to2022.csv
Mareca strepera -> /content/checklist/ebird_filtered_checklist/^Mareca strepera_filtered_2019to2022.csv
Motacilla alba -> /content/ch

In [ ]:
!pip install gpboost
import gpboost as gpb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 17.9 MB/s eta 0:00:00


In [ ]:
def _parameters_to_summary_df(pars, birdname, source, component_names=None):
    """Convert GPBoost covariance/auxiliary parameter objects into a conservative long-ish CSV table."""
    component_names = component_names or []

    if pars is None:
        return pd.DataFrame()

    if isinstance(pars, pd.DataFrame):
        df_pars = pars.copy().reset_index()
    elif isinstance(pars, pd.Series):
        df_pars = pars.to_frame(name="value").reset_index()
    elif isinstance(pars, dict):
        df_pars = pd.DataFrame(list(pars.items()), columns=["parameter", "value"])
    else:
        arr = np.asarray(pars).reshape(-1)
        df_pars = pd.DataFrame({"parameter_index": range(len(arr)), "value": arr})

    df_pars.insert(0, "birdname", birdname)
    df_pars.insert(1, "parameter_source", source)

    if "component" not in df_pars.columns:
        df_pars.insert(
            2,
            "component",
            [component_names[i] if i < len(component_names) else "" for i in range(len(df_pars))]
        )

    return df_pars


def _predict_response_mean(model, data, group_data_pred=None, ignore_gp_model=False):
    if ignore_gp_model:
        pred = model.predict(data=data, ignore_gp_model=True)
    else:
        pred = model.predict(
            data=data,
            group_data_pred=group_data_pred,
            predict_response=True
        )

    if isinstance(pred, dict):
        if "response_mean" in pred:
            pred = pred["response_mean"]
        elif "mu" in pred:
            pred = pred["mu"]
        else:
            first_key = list(pred.keys())[0]
            pred = pred[first_key]

    pred = np.asarray(pred).reshape(-1)

    if RESPONSE_TRANSFORM == "log1p" and GPBOOST_LIKELIHOOD == "gaussian":
        return np.expm1(pred)

    return pred


def _make_response(y):
    y = pd.to_numeric(y, errors="coerce")
    if RESPONSE_TRANSFORM == "log1p":
        return np.log1p(y)
    return y


def run_gpboost_for_bird(birdname, checklist_path, env, output_root=OUTPUT_ROOT):
    bird_slug = safe_filename(birdname)
    bird_output_dir = os.path.join(output_root, bird_slug)
    os.makedirs(bird_output_dir, exist_ok=True)

    print("\n" + "=" * 70)
    print("Running bird:", birdname)
    print("Checklist:", checklist_path)
    print("Output:", bird_output_dir)

    # ============================================================
    # Read and preprocess checklist
    # ============================================================
    checklist = pd.read_csv(checklist_path)

    checklist["observation_date"] = pd.to_datetime(checklist["observation_date"])
    checklist["year_number"] = checklist["observation_date"].dt.year
    checklist["month_number"] = checklist["observation_date"].dt.month

    checklist["day_of_year"] = checklist["observation_date"].dt.dayofyear
    checklist["doy_sin"] = np.sin(2 * np.pi * checklist["day_of_year"] / 365.25)
    checklist["doy_cos"] = np.cos(2 * np.pi * checklist["day_of_year"] / 365.25)

    checklist["start_hour"] = pd.to_datetime(
        checklist["time_started"],
        format="%H:%M:%S"
    ).dt.hour
    checklist["start_hour_sin"] = np.sin(2 * np.pi * checklist["start_hour"] / 24)
    checklist["start_hour_cos"] = np.cos(2 * np.pi * checklist["start_hour"] / 24)

    time = pd.to_datetime(checklist["time_started"], format="%H:%M:%S")
    checklist["start_minutes"] = (
        time.dt.hour * 60 + time.dt.minute + time.dt.second / 60
    )
    checklist["cross_day"] = (
        checklist["start_minutes"] + checklist["duration_mins"] >= 1440
    ).astype(int)
    checklist["end_minutes"] = (
        checklist["start_minutes"] + checklist["duration_mins"]
    ) % 1440
    checklist["end_hour"] = checklist["end_minutes"] / 60

    cl_exclude_cols = [
        "category", "common_name", "observation_date", "all_species_reported",
        "group_identifier", "time_started"
    ]
    cl_cols = [col for col in checklist.columns if col not in cl_exclude_cols]
    checklist = checklist[cl_cols].copy()

    checklist["observation_count"] = pd.to_numeric(
        checklist["observation_count"].replace("X", 0),
        errors="coerce"
    )
    checklist["locality_type"] = (
        checklist["locality_type"]
        .map({"P": 0, "H": 1})
        .astype("Int64")
    )

    protocol_cols = ["Traveling", "Stationary"]
    checklist = checklist[checklist["protocol_type"].isin(protocol_cols)].copy()
    checklist["protocol_type"] = (
        checklist["protocol_type"]
        .map({"Stationary": 0, "Traveling": 1})
        .astype(int)
    )
    checklist = checklist[checklist["protocol_type"] == 1].copy()

    print("length of checklist:", len(checklist))
    for year in [2019, 2020, 2021, 2022]:
        print(f"length of checklist in {year}:", len(checklist[checklist["year_number"] == year]))

    # ============================================================
    # Merge environmental data and apply original filters
    # ============================================================
    df = pd.merge(
        checklist,
        env,
        on=["Id", "month_number", "year_number"],
        how="inner"
    )
    print("Before dropna:", len(df))

    df = df.dropna().reset_index(drop=True)
    print("After dropna:", len(df))

    if len(df) == 0:
        raise ValueError(f"{birdname}: merge/dropna ?????")

    oc_q99 = df["observation_count"].quantile(0.99)
    df = df[df["observation_count"] <= oc_q99].copy()
    print("99% cutoff of observation count:", oc_q99)
    print("rows after cut:", len(df))

    observer_info = pd.DataFrame(df.groupby("observer_id").size().reset_index())
    observer_info.columns = ["observer_id", "count"]
    orc_q70 = observer_info["count"].quantile(0.7)
    observer_list = observer_info["observer_id"][observer_info["count"] > orc_q70].copy()
    df = df[df["observer_id"].isin(observer_list)]
    print(f"70% cutoff of observer count: {orc_q70}")
    print("number of observers:", len(observer_list))
    print("rows after cut:", len(df))

    id_info = pd.DataFrame(df.groupby("Id").size().reset_index())
    id_info.columns = ["Id", "count"]
    idc_q50 = id_info["count"].quantile(0.5)
    id_list = id_info["Id"][id_info["count"] > idc_q50]
    df = df[df["Id"].isin(id_list)].copy()
    print(f"50% cutoff of Id count: {idc_q50}")
    print("number of Ids:", len(id_list))
    print("rows after cut:", len(df))

    if df["Id"].nunique() < 2:
        raise ValueError(f"{birdname}: ??? Id ??????? validation set")

    # ============================================================
    # Train/validation split by Id
    # ============================================================
    df["sample_id"] = range(1, len(df) + 1)

    rng = np.random.default_rng(TRAIN_SEED)

    val_idx = rng.choice(
        df.index,
        size=max(1, int(len(df) * 0.1)),
        replace=False
    )

    val_df = df.loc[val_idx].copy()
    train_df = df.drop(index=val_idx).copy()

    print("all samples:", len(df))
    print("Validation samples:", len(val_df))
    print("Train samples:", len(train_df))
    print("Train Id:", train_df["Id"].nunique())
    print("Validation Id:", val_df["Id"].nunique())
    print("Validation rows:", len(val_df))
    print("Train rows:", len(train_df))

    model_include_cols = [
        "scientific_name", "observer_id", "Id", "month_number", "year_number",
        "sample_id", "observation_count", "day_of_year", "start_hour",
        "start_minutes", "end_minutes", "end_hour", "protocol_type", "cross_day",
        "start_hour_cos", "start_hour_sin", "number_observers", "longitude", "latitude",
         "Id", "doy_sin", "doy_cos"
    ]
    model_cols = [col for col in df.columns if col not in model_include_cols]

    x_train = train_df[model_cols].copy()
    y_train = _make_response(train_df["observation_count"])

    x_val = val_df[model_cols].copy()
    y_val = _make_response(val_df["observation_count"])

    #group_train = train_df[["observer_id", "Id"]].copy()
    #group_val = val_df[["observer_id", "Id"]].copy()

    group_train = train_df[["observer_id"]].copy()
    group_val = val_df[["observer_id"]].copy()

    assert len(x_train) == len(y_train) == len(group_train)
    assert len(x_val) == len(y_val) == len(group_val)

    print("Train:", x_train.shape)
    print("Val:", x_val.shape)
    print("Train observers:", group_train["observer_id"].nunique())
    #print("Train Id:", group_train["Id"].nunique())
    print("Val observers:", group_val["observer_id"].nunique())
    #print("Val Id:", group_val["Id"].nunique())

    print("X train NA:", x_train.isna().sum().sum())
    print("X val NA:", x_val.isna().sum().sum())
    print("Y train NA:", pd.isna(y_train).sum())
    print("Y val NA:", pd.isna(y_val).sum())

    X_train_np = x_train.to_numpy(dtype=np.float64)
    X_val_np = x_val.to_numpy(dtype=np.float64)

    y_train_np = np.asarray(y_train, dtype=np.float64)
    y_val_np = np.asarray(y_val, dtype=np.float64)

    group_train_np = group_train.astype(str).to_numpy()
    group_val_np = group_val.astype(str).to_numpy()

    # ============================================================
    # GPBoost model: preserve notebook likelihood/parameters
    # ============================================================
    gp_model = gpb.GPModel(
        group_data=group_train_np,
        likelihood=GPBOOST_LIKELIHOOD
    )

    train_data = gpb.Dataset(
        data=X_train_np,
        label=y_train_np,
        feature_name=list(x_train.columns)
    )

    model = gpb.train(
        params=GPBOOST_PARAMS,
        train_set=train_data,
        gp_model=gp_model,
        num_boost_round=NUM_BOOST_ROUND
    )

    # ============================================================
    # Random-effect variance/covariance parameters
    # ============================================================
    cov_pars = gp_model.get_cov_pars()
    aux_pars = None
    try:
        aux_pars = gp_model.get_aux_pars()
    except Exception as err:
        print("get_aux_pars unavailable:", err)

    re_train_available = False
    re_train_note = ""
    try:
        re_train = model.predict_training_data_random_effects()
        re_train_available = True
        re_train_note = "predict_training_data_random_effects() available; individual RE not saved in summary CSV"
        print(re_train.head())
        print(re_train.shape)
    except Exception as err:
        re_train_note = (
            "predict_training_data_random_effects() unavailable for this fitted model/API; "
            "saved variance components only, not individual conditional random effects. "
            f"Original error: {err}"
        )
        print(re_train_note)

    random_summary_parts = [
        _parameters_to_summary_df(
            cov_pars,
            birdname=birdname,
            source="covariance_or_variance_parameters",
            component_names=["observer"] #, "Id"
        )
    ]
    aux_df = _parameters_to_summary_df(
        aux_pars,
        birdname=birdname,
        source="auxiliary_parameters",
        component_names=["error_variance_or_likelihood_auxiliary"]
    )
    if len(aux_df) > 0:
        random_summary_parts.append(aux_df)

    random_effect_summary_df = pd.concat(
        random_summary_parts,
        ignore_index=True,
        sort=False
    )
    random_effect_summary_df["individual_random_effects_available"] = re_train_available
    random_effect_summary_df["note"] = re_train_note

    random_effect_path = os.path.join(
        bird_output_dir,
        f"{bird_slug}_random_effect_summary.csv"
    )
    random_effect_summary_df.to_csv(random_effect_path, index=False)

    # ============================================================
    # Model performance
    # ============================================================
    y_pred_val = _predict_response_mean(
        model,
        data=X_val_np,
        group_data_pred=group_val_np,
        ignore_gp_model=False
    )
    y_pred_train = _predict_response_mean(
        model,
        data=X_train_np,
        group_data_pred=group_train_np,
        ignore_gp_model=False
    )

    y_train_eval = train_df["observation_count"].to_numpy(dtype=np.float64)
    y_val_eval = val_df["observation_count"].to_numpy(dtype=np.float64)

    if RESPONSE_TRANSFORM == "log1p" and GPBOOST_LIKELIHOOD == "gaussian":
        # predictions already back-transformed by _predict_response_mean().
        pass

    src_val, _ = spearmanr(y_val_eval, y_pred_val)
    src_train, _ = spearmanr(y_train_eval, y_pred_train)
    mae_val = mean_absolute_error(y_val_eval, y_pred_val)
    mae_train = mean_absolute_error(y_train_eval, y_pred_train)

    performance_df = pd.DataFrame([{
        "birdname": birdname,
        "checklist_path": checklist_path,
        "response_transform": RESPONSE_TRANSFORM,
        "gpboost_likelihood": GPBOOST_LIKELIHOOD,
        "n_train": len(train_df),
        "n_val": len(val_df),
        "n_train_id": train_df["Id"].nunique(),
        "n_val_id": val_df["Id"].nunique(),
        "train Spearman SRC": src_train,
        "validation Spearman SRC": src_val,
        "train MAE": mae_train,
        "validation MAE": mae_val,
        "train_spearman_src": src_train,
        "validation_spearman_src": src_val,
        "train_mae": mae_train,
        "validation_mae": mae_val,
    }])

    performance_path = os.path.join(
        bird_output_dir,
        f"{bird_slug}_model_performance.csv"
    )
    performance_df.to_csv(performance_path, index=False)

    print("Train SRC:", src_train)
    print("Validation SRC:", src_val)
    print("Train MAE:", mae_train)
    print("Validation MAE:", mae_val)

    # ============================================================
    # Reconstruction: preserve standardized conditions + ignore_gp_model
    # ============================================================
    valid_ids = df["Id"].unique()
    recon_df = env[env["Id"].isin(valid_ids)].copy()
    recon_df = recon_df[recon_df["year_number"].isin([2021, 2022])].copy()

    print("df unique Id:", df["Id"].nunique())
    print("recon unique Id:", recon_df["Id"].nunique())
    print("recon rows:", len(recon_df))

    reference_duration = train_df["duration_mins"].median()
    reference_effort = train_df["effort_km"].median()
    reference_locality = train_df["locality_type"].mode()[0]

    recon_df["duration_mins"] = reference_duration
    recon_df["effort_km"] = reference_effort
    recon_df["locality_type"] = reference_locality

    recon_df["date"] = pd.to_datetime(
        dict(
            year=recon_df["year_number"],
            month=recon_df["month_number"],
            day=15
        )
    )
    recon_df["day_of_year"] = recon_df["date"].dt.dayofyear
    #recon_df["doy_sin"] = np.sin(2 * np.pi * recon_df["day_of_year"] / 365.25)
    #recon_df["doy_cos"] = np.cos(2 * np.pi * recon_df["day_of_year"] / 365.25)

    missing_cols = [col for col in model_cols if col not in recon_df.columns]
    if len(missing_cols) > 0:
        raise ValueError(f"recon_df ???? model features: {missing_cols}")

    X_recon = recon_df[model_cols].copy()

    print("Number of reconstruction samples:", len(X_recon))
    print("Number of features:", X_recon.shape[1])
    print("Feature order same as training:", list(X_recon.columns) == list(x_train.columns))

    if list(X_recon.columns) != list(x_train.columns):
        raise ValueError("X_recon feature columns/order ? x_train ???")

    print("NA count:", X_recon.isna().sum().sum())
    numeric_recon = X_recon.select_dtypes(include=[np.number])
    print("Inf count:", np.isinf(numeric_recon.to_numpy()).sum())

    X_recon_np = X_recon.to_numpy(dtype=np.float64)

    #recon_df["observer_id"] = "__new_observer__"

    #group_recon_np = (
    #    recon_df[["observer_id", "Id"]]
    #    .astype(str)
    #    .to_numpy()
    #)

    #pred_recon = _predict_response_mean(
    #    model,
    #    data=X_recon_np,
    #    group_data_pred=group_recon_np,
    #    ignore_gp_model=False
    #)

    pred_recon = _predict_response_mean(
        model,
        data=X_recon_np,
        ignore_gp_model=True
    )
    recon_df["birdname"] = birdname
    recon_df["relative_abundance"] = pred_recon
    recon_df["reference_duration_mins"] = reference_duration
    recon_df["reference_effort_km"] = reference_effort
    recon_df["reference_locality_type"] = reference_locality

    reconstruction_path = os.path.join(
        bird_output_dir,
        f"{bird_slug}_reconstruction_abundance.csv"
    )
    recon_df.to_csv(reconstruction_path, index=False)

    print("===== Reconstruction summary =====")
    print(recon_df["relative_abundance"].describe())

    # ============================================================
    # Feature importance: gain normalized to proportion
    # ============================================================
    gain = model.feature_importance(importance_type="gain")
    gain_sum = np.sum(gain)
    if gain_sum > 0:
        importance_prop = gain / gain_sum
    else:
        importance_prop = np.zeros_like(gain, dtype=float)

    importance_df = pd.DataFrame({
        "birdname": birdname,
        "feature": model.feature_name(),
        "gain": gain,
        "importance": importance_prop,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    feature_importance_path = os.path.join(
        bird_output_dir,
        f"{bird_slug}_feature_importance_gain.csv"
    )
    importance_df.to_csv(feature_importance_path, index=False)

    print("Top feature importance:")
    print(importance_df.head(20))

    return {
        "birdname": birdname,
        "output_dir": bird_output_dir,
        "reconstruction_path": reconstruction_path,
        "feature_importance_path": feature_importance_path,
        "performance_path": performance_path,
        "random_effect_path": random_effect_path,
        "performance_df": performance_df,
        "feature_importance_df": importance_df,
        "random_effect_summary_df": random_effect_summary_df,
    }


In [ ]:
# ============================================================
# Run batch and save/download after each bird
# ============================================================
def make_bird_zip(bird_output_dir, bird_slug, output_root=OUTPUT_ROOT):
    zip_base = os.path.join(output_root, f"{bird_slug}_outputs")
    zip_path = shutil.make_archive(
        base_name=zip_base,
        format="zip",
        root_dir=bird_output_dir,
    )
    return zip_path


def maybe_download_file(path):
    try:
        from google.colab import files
        files.download(path)
    except Exception as err:
        print("Download skipped. File saved at:", path)
        print("Reason:", err)


def write_completed_marker(bird_output_dir, birdname, zip_path):
    marker_path = os.path.join(bird_output_dir, "_COMPLETED.txt")
    with open(marker_path, "w", encoding="utf-8") as f:
        f.write(f"birdname={birdname}\n")
        f.write(f"zip_path={zip_path}\n")
        f.write(f"completed_at={pd.Timestamp.now()}\n")
    return marker_path


def rebuild_summary_csvs_from_disk(output_root=OUTPUT_ROOT):
    performance_files = []
    importance_files = []
    random_effect_files = []
    failed_file = os.path.join(output_root, "failed_birds.csv")

    for root, dirs, files_in_dir in os.walk(output_root):
        for file in files_in_dir:
            path = os.path.join(root, file)
            if file.endswith("_model_performance.csv"):
                performance_files.append(path)
            elif file.endswith("_feature_importance_gain.csv"):
                importance_files.append(path)
            elif file.endswith("_random_effect_summary.csv"):
                random_effect_files.append(path)

    if len(performance_files) > 0:
        pd.concat(
            [pd.read_csv(p) for p in sorted(performance_files)],
            ignore_index=True
        ).to_csv(os.path.join(output_root, "all_model_performance.csv"), index=False)

    if len(importance_files) > 0:
        pd.concat(
            [pd.read_csv(p) for p in sorted(importance_files)],
            ignore_index=True
        ).to_csv(os.path.join(output_root, "all_feature_importance_gain.csv"), index=False)

    if len(random_effect_files) > 0:
        pd.concat(
            [pd.read_csv(p) for p in sorted(random_effect_files)],
            ignore_index=True,
            sort=False
        ).to_csv(os.path.join(output_root, "all_random_effect_summary.csv"), index=False)

    return {
        "performance_files": performance_files,
        "importance_files": importance_files,
        "random_effect_files": random_effect_files,
        "failed_file": failed_file,
    }


all_results = []
failed_results = []
skipped_birds = []

for birdname, checklist_path in bird_path_map.items():
    bird_slug = safe_filename(birdname)
    bird_output_dir = os.path.join(OUTPUT_ROOT, bird_slug)
    expected_zip_path = os.path.join(OUTPUT_ROOT, f"{bird_slug}_outputs.zip")
    completed_marker = os.path.join(bird_output_dir, "_COMPLETED.txt")

    if (
        SKIP_FINISHED_BIRDS
        and os.path.exists(completed_marker)
        and os.path.exists(expected_zip_path)
    ):
        print(f"\nSkip {birdname}: already completed.")
        print("Existing zip:", expected_zip_path)
        skipped_birds.append({
            "birdname": birdname,
            "checklist_path": checklist_path,
            "zip_path": expected_zip_path,
        })
        continue

    try:
        result = run_gpboost_for_bird(
            birdname=birdname,
            checklist_path=checklist_path,
            env=env,
            output_root=OUTPUT_ROOT,
        )
        all_results.append(result)

        # ???????????????????????? zip?
        rebuild_summary_csvs_from_disk(OUTPUT_ROOT)

        if ZIP_EACH_BIRD:
            bird_zip_path = make_bird_zip(
                bird_output_dir=result["output_dir"],
                bird_slug=safe_filename(result["birdname"]),
                output_root=OUTPUT_ROOT,
            )
            write_completed_marker(
                bird_output_dir=result["output_dir"],
                birdname=result["birdname"],
                zip_path=bird_zip_path,
            )
            print("Saved bird zip:", bird_zip_path)

            if DOWNLOAD_EACH_BIRD_ZIP:
                maybe_download_file(bird_zip_path)

        # ???? zip??????????????? zip?

    except Exception as err:
        print(f"ERROR while running {birdname}: {err}")
        failed_results.append({
            "birdname": birdname,
            "checklist_path": checklist_path,
            "error": str(err),
        })
        pd.DataFrame(failed_results).to_csv(
            os.path.join(OUTPUT_ROOT, "failed_birds.csv"),
            index=False
        )

# ??????????????? zip?
rebuild_summary_csvs_from_disk(OUTPUT_ROOT)

if len(failed_results) > 0:
    pd.DataFrame(failed_results).to_csv(
        os.path.join(OUTPUT_ROOT, "failed_birds.csv"),
        index=False
    )

if len(skipped_birds) > 0:
    pd.DataFrame(skipped_birds).to_csv(
        os.path.join(OUTPUT_ROOT, "skipped_birds.csv"),
        index=False
    )

print("\nFinished birds this run:", len(all_results))
print("Skipped birds:", len(skipped_birds))
print("Failed birds:", len(failed_results))
print("No final all-results zip was created. Each completed bird has its own zip file.")



Running bird: Ichthyaetus melanocephalus
Checklist: /content/checklist/ebird_filtered_checklist/^Ichthyaetus melanocephalus_filtered_2019to2022.csv
Output: /content/drive/MyDrive/gpboost_batch_outputs/Ichthyaetus_melanocephalus
length of checklist: 33644
length of checklist in 2019: 5530
length of checklist in 2020: 6135
length of checklist in 2021: 9372
length of checklist in 2022: 12607
Before dropna: 21979
After dropna: 21913
99% cutoff of observation count: 263.76000000000204
rows after cut: 21693
70% cutoff of observer count: 4.0
number of observers: 1082
rows after cut: 16149
50% cutoff of Id count: 7.0
number of Ids: 158
rows after cut: 15705
all samples: 15705
Validation samples: 1570
Train samples: 14135
Train Id: 158
Validation Id: 144
Validation rows: 1570
Train rows: 14135
Train: (14135, 71)
Val: (1570, 71)
Train observers: 1074
Val observers: 682
X train NA: 0
X val NA: 0
Y train NA: 0
Y val NA: 0
    Group_1
0 -0.440595
1  0.232259
2 -0.513720
3  0.730572
4 -0.204447
(14